# 07 - Additional Faithfulness And Class-Specificity Evaluation

This notebook first runs a small executable smoke test and can then run the same evaluation over every correctly classified example for the attribution methods presented in the report:

- `level3`
- `legrad_final_score_relu_attention_gradient_mean_layers_source_tokens`
- `legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens`
- `legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens`
- `legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens`

Use `RUN_FULL=False` to verify the pipeline on a few utterances. After that succeeds in Colab, set `RUN_FULL=True` to run the additional faithfulness and class-specificity evaluation over the complete prediction manifest.

In [1]:
from pathlib import Path
import subprocess
import sys
import zipfile

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Colab workflow: keep RUN_FULL=False for a quick test, then change it to True.
RUN_FULL = True
SMOKE_EXAMPLES = 2
DATASETS = ["RAVDESS"]  # Use ["IEMOCAP", "RAVDESS"] for both datasets.
PREDICTIONS_CSV = None  # Full Colab path to predictions.csv/original_predictions.csv, if available.
IS_COLAB = "google.colab" in sys.modules
DEVICE = "cuda" if IS_COLAB else "cpu"
OUTPUT_ROOT = (
    Path("/content/drive/MyDrive/gradient_based_speech_xai_outputs/test_07_additional_evaluation")
    if IS_COLAB
    else ROOT / "outputs" / "test_07_additional_evaluation"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REPORT_METHODS = [
    "level3",
    "legrad_final_score_relu_attention_gradient_mean_layers_source_tokens",
    "legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens",
    "legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens",
    "legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens",
]
METHODS_ARG = ",".join(REPORT_METHODS)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
print("Project root:", ROOT)
print("Output root:", OUTPUT_ROOT)
print("Run scope:", "ALL correct examples" if RUN_FULL else f"smoke test ({SMOKE_EXAMPLES} examples)")
print("Device:", DEVICE)
print("Datasets:", DATASETS)
print("Methods:")
for method in REPORT_METHODS:
    print("-", method)


Project root: C:\Users\mateu\repos\gradient_based_speach_xai
Output root: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation
Run scope: smoke test (2 examples)
Device: cpu
Methods:
- level3
- legrad_final_score_relu_attention_gradient_mean_layers_source_tokens
- legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens
- legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens
- legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens


## Select The Prediction Manifest

The evaluation scripts require a prediction CSV with valid `audio_path` values. On Colab, set `PREDICTIONS_CSV` above to the complete `original_predictions.csv` or `predictions.csv` produced by the preceding evaluation. The ZIP fallback below remains available for the local smoke test.

In [2]:
def local_audio_path(row: pd.Series) -> str | None:
    relative_path = str(row["relative_path"]).replace("\\", "/")
    dataset = str(row["dataset"])
    if dataset == "IEMOCAP":
        candidates = [ROOT / "data" / relative_path]
    elif dataset == "RAVDESS":
        candidates = [
            ROOT / "data" / "ravdess" / relative_path,
            ROOT / "data" / "ravdess" / "Audio_Speech_Actors_01-24" / relative_path,
        ]
    else:
        candidates = [ROOT / relative_path]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return None


def build_smoke_predictions() -> Path:
    sources = [
        (
            ROOT / "IEMOCAP_speech_xai_results_20260711_074459.zip",
            "iemocap_ravdess_duration_matched_speechxai_20260710_194505_901788/duration_matched_records.csv",
        ),
        (
            ROOT / "RAVDESS_speech_xai_results_20260711_100706.zip",
            "ravdess_duration_matched_speechxai_20260711_092556_063830/duration_matched_records.csv",
        ),
    ]
    usecols = [
        "dataset", "audio_path", "audio_id", "relative_path", "true_class", "true_label",
        "session_id", "dialogue_id", "utterance_id", "iemocap_emotion", "annotation_path",
        "actor_id", "ravdess_emotion", "original_class", "original_label", "original_confidence",
    ]
    frames = []
    for zip_path, inner_path in sources:
        if not zip_path.exists():
            raise FileNotFoundError(zip_path)
        with zipfile.ZipFile(zip_path) as archive:
            with archive.open(inner_path) as handle:
                header = pd.read_csv(handle, nrows=0).columns.tolist()
                columns = [column for column in usecols if column in header]
            with archive.open(inner_path) as handle:
                frame = pd.read_csv(handle, usecols=columns, low_memory=False)
        frame = frame[frame["original_class"].eq(frame["true_class"])].copy()
        frame = frame.drop_duplicates(["dataset", "audio_id"])
        frames.append(frame)

    records = pd.concat(frames, ignore_index=True)
    records = records[records["dataset"].isin(DATASETS)].copy()
    records["audio_path"] = records.apply(local_audio_path, axis=1)
    records = records[records["audio_path"].notna()].copy()
    records["predicted_class"] = records["original_class"].astype(int)
    records["predicted_label"] = records["original_label"].astype(str)
    records["confidence"] = records["original_confidence"].astype(float)
    records["is_correct"] = records["predicted_class"].eq(records["true_class"].astype(int))

    if RUN_FULL:
        selected = records.sort_values(["dataset", "true_class", "audio_id"]).reset_index(drop=True)
    else:
        selected_parts = []
        for dataset in ["IEMOCAP", "RAVDESS"]:
            subset = records[records["dataset"].eq(dataset)].copy()
            for class_id in sorted(subset["true_class"].unique()):
                selected_parts.append(subset[subset["true_class"].eq(class_id)].sort_values("audio_id").head(1))
        selected = pd.concat(selected_parts, ignore_index=True).head(8)

    base_columns = [
        "dataset", "audio_path", "relative_path", "audio_id", "true_class", "true_label",
        "predicted_class", "predicted_label", "is_correct", "confidence", "session_id",
        "dialogue_id", "utterance_id", "iemocap_emotion", "annotation_path", "actor_id",
        "ravdess_emotion",
    ]
    selected = selected[[column for column in base_columns if column in selected.columns]]
    output_path = OUTPUT_ROOT / ("report_methods_full_predictions.csv" if RUN_FULL else "report_methods_smoke_predictions.csv")
    selected.to_csv(output_path, index=False)
    return output_path

predictions_csv = Path(PREDICTIONS_CSV).expanduser() if PREDICTIONS_CSV else build_smoke_predictions()
if not predictions_csv.exists():
    raise FileNotFoundError(f"Prediction manifest not found: {predictions_csv}")
predictions = pd.read_csv(predictions_csv)
print(predictions_csv)
display(predictions[["dataset", "audio_id", "true_label", "predicted_label", "audio_path"]])
print("Rows:", len(predictions), "Correct:", int(predictions["is_correct"].sum()))


C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\report_methods_smoke_predictions.csv


,dataset,audio_id,true_label,predicted_label,audio_path
0,IEMOCAP,Ses01F_impro02_M015,neu,neu,C:\Users\mateu\repos\gradient_based_speach_xai...
1,IEMOCAP,Ses01F_impro03_F000,hap,hap,C:\Users\mateu\repos\gradient_based_speach_xai...
2,IEMOCAP,Ses01F_impro01_F012,ang,ang,C:\Users\mateu\repos\gradient_based_speach_xai...
3,IEMOCAP,Ses01F_impro02_F000,sad,sad,C:\Users\mateu\repos\gradient_based_speach_xai...
4,RAVDESS,03-01-01-01-01-01-09,neu,neu,C:\Users\mateu\repos\gradient_based_speach_xai...
5,RAVDESS,03-01-03-01-01-01-03,hap,hap,C:\Users\mateu\repos\gradient_based_speach_xai...
6,RAVDESS,03-01-05-01-01-01-01,ang,ang,C:\Users\mateu\repos\gradient_based_speach_xai...
7,RAVDESS,03-01-04-01-01-01-05,sad,sad,C:\Users\mateu\repos\gradient_based_speach_xai...


Rows: 8 Correct: 8


## Run Deletion-Faithfulness Smoke Test

Smoke mode uses a few correctly classified audios, one deletion fraction, one random trial, and all report methods. Full mode uses every correct prediction, four deletion fractions, and three random trials.

In [3]:
def run_command(command: list[str]) -> None:
    print(" ".join(str(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")


scope_args = ["--all-examples"] if RUN_FULL else ["--max-examples", str(SMOKE_EXAMPLES)]
download_args = [] if IS_COLAB else ["--local-files-only"]
dataset_label = "_".join(dataset.lower() for dataset in DATASETS)
run_label = f"{dataset_label}_report_methods_full" if RUN_FULL else f"{dataset_label}_report_methods_smoke"
deletion_root = OUTPUT_ROOT / "deletion_faithfulness"
run_command([
    sys.executable,
    "scripts/evaluate_deletion_faithfulness.py",
    "--predictions-csv", str(predictions_csv),
    "--dataset-name", run_label,
    *scope_args,
    "--fractions", "0.05,0.1,0.2,0.3" if RUN_FULL else "0.1",
    "--random-trials", "3" if RUN_FULL else "1",
    "--modes", METHODS_ARG,
    "--device", DEVICE,
    *download_args,
    "--output-root", str(deletion_root),
])

deletion_runs = sorted(deletion_root.glob(f"{run_label}_deletion_faithfulness_*"), key=lambda path: path.stat().st_mtime)
deletion_run = deletion_runs[-1]
print("Deletion run:", deletion_run)


C:\Users\mateu\repos\gradient_based_speach_xai\.venv\Scripts\python.exe scripts/evaluate_deletion_faithfulness.py --predictions-csv C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\report_methods_smoke_predictions.csv --dataset-name report_methods_smoke --max-examples 2 --fractions 0.1 --random-trials 1 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cpu --local-files-only --output-root C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\deletion_faithfulness


Explained 1/2: 03-01-01-01-01-01-09.wav
Explained 2/2: 03-01-03-01-01-01-03.wav

Deletion-faithfulness evaluation complete
Examples: 2
Records: 30
Sparsity records: 10
Saved new run to: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\deletion_faithfulness\report_methods_smoke_deletion_faithfulness_20260712_105750_662610

Deletion run: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\deletion_faithfulness\report_methods_smoke_deletion_faithfulness_20260712_105750_662610


In [4]:
deletion_records = pd.read_csv(deletion_run / "deletion_records.csv")
delection_summary = pd.read_csv(deletion_run / "deletion_summary.csv")
sparsity_summary = pd.read_csv(deletion_run / "sparsity_summary_by_method.csv")
print("Deletion records:", len(deletion_records))
display(delection_summary[["explanation_mode", "strategy", "fraction", "mean_logit_drop", "mean_probability_drop"]])
display(sparsity_summary[["explanation_mode", "examples", "mean_normalized_entropy", "mean_gini", "mean_top_10_percent_mass"]])


Deletion records: 30


,explanation_mode,strategy,fraction,mean_logit_drop,mean_probability_drop
0,legrad_final_score_relu_attention_gradient_mea...,bottom,0.1,-0.084502,-0.019956
1,legrad_final_score_relu_attention_gradient_mea...,random,0.1,-0.158386,-0.080106
2,legrad_final_score_relu_attention_gradient_mea...,top,0.1,0.128331,0.058208
3,legrad_final_score_relu_attention_gradient_ren...,bottom,0.1,-0.084502,-0.019956
4,legrad_final_score_relu_attention_gradient_ren...,random,0.1,-0.158386,-0.080106
5,legrad_final_score_relu_attention_gradient_ren...,top,0.1,0.146179,0.077618
6,legrad_layer_local_score_relu_attention_gradie...,bottom,0.1,0.032183,0.027138
7,legrad_layer_local_score_relu_attention_gradie...,random,0.1,-0.158386,-0.080106
8,legrad_layer_local_score_relu_attention_gradie...,top,0.1,0.177257,0.056595
9,legrad_layer_local_score_relu_attention_gradie...,bottom,0.1,0.023220,0.020222


,explanation_mode,examples,mean_normalized_entropy,mean_gini,mean_top_10_percent_mass
0,legrad_final_score_relu_attention_gradient_mea...,2,0.999228,0.050655,0.119331
1,legrad_final_score_relu_attention_gradient_ren...,2,0.999214,0.051115,0.119388
2,legrad_layer_local_score_relu_attention_gradie...,2,0.998178,0.076331,0.128162
3,legrad_layer_local_score_relu_attention_gradie...,2,0.998149,0.076903,0.128393
4,level3,2,0.874677,0.581122,0.329561


## Run Class-Specificity Smoke Test

For every selected audio, this computes relevance for the model prediction and for the runner-up class. Lower predicted-vs-runner-up correlation means the explanation changes more when the explained class changes.

In [5]:
class_specificity_root = OUTPUT_ROOT / "class_specificity"
run_command([
    sys.executable,
    "scripts/evaluate_class_specificity.py",
    "--predictions-csv", str(predictions_csv),
    "--dataset-name", run_label,
    *scope_args,
    "--modes", METHODS_ARG,
    "--device", DEVICE,
    *download_args,
    "--output-root", str(class_specificity_root),
])

class_specificity_runs = sorted(class_specificity_root.glob(f"{run_label}_class_specificity_*"), key=lambda path: path.stat().st_mtime)
class_specificity_run = class_specificity_runs[-1]
print("Class-specificity run:", class_specificity_run)


C:\Users\mateu\repos\gradient_based_speach_xai\.venv\Scripts\python.exe scripts/evaluate_class_specificity.py --predictions-csv C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\report_methods_smoke_predictions.csv --dataset-name report_methods_smoke --max-examples 2 --modes level3,legrad_final_score_relu_attention_gradient_mean_layers_source_tokens,legrad_final_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens,legrad_layer_local_score_relu_attention_gradient_mean_layers_source_tokens,legrad_layer_local_score_relu_attention_gradient_renormalized_hubert_layer_weighted_source_tokens --device cpu --local-files-only --output-root C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\class_specificity


Compared 1/2: 03-01-01-01-01-01-09.wav
Compared 2/2: 03-01-03-01-01-01-03.wav

Class-specificity evaluation complete
Examples: 2
Records: 10
Saved new run to: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\class_specificity\report_methods_smoke_class_specificity_20260712_105818_610561

Class-specificity run: C:\Users\mateu\repos\gradient_based_speach_xai\outputs\test_07_additional_evaluation\class_specificity\report_methods_smoke_class_specificity_20260712_105818_610561


In [6]:
class_specificity_records = pd.read_csv(class_specificity_run / "class_specificity_records.csv")
class_specificity_summary = pd.read_csv(class_specificity_run / "class_specificity_summary_by_method.csv")
print("Class-specificity records:", len(class_specificity_records))
display(class_specificity_summary[["explanation_mode", "examples", "mean_pearson_correlation", "mean_spearman_correlation"]])


Class-specificity records: 10


,explanation_mode,examples,mean_pearson_correlation,mean_spearman_correlation
0,legrad_final_score_relu_attention_gradient_mea...,2,0.784355,0.786529
1,legrad_final_score_relu_attention_gradient_ren...,2,0.782747,0.786834
2,legrad_layer_local_score_relu_attention_gradie...,2,0.415282,0.416854
3,legrad_layer_local_score_relu_attention_gradie...,2,0.428063,0.426573
4,level3,2,-0.272416,-0.371413


## Smoke Test, Then Full Colab Evaluation

With `RUN_FULL=False`, the notebook evaluates `SMOKE_EXAMPLES` correctly classified utterances. Verify the generated tables, then set `RUN_FULL=True` and rerun the deletion and class-specificity sections. Full mode passes `--all-examples`, uses four deletion fractions and three random trials, runs on CUDA in Colab, and writes timestamped results to Google Drive.

For each evaluated utterance in smoke mode:

- deletion-faithfulness produces `15` rows: five methods times top/bottom/random.
- sparsity produces `5` rows: one per method.
- class-specificity produces `5` rows: one per method.

These are smoke-test counts only. They verify that the report attribution methods execute end-to-end; the full quantitative comparison remains the duration-matched Pastor/SpeechXAI evaluation.